In [2]:
!pip install -q timm scikit-learn opencv-python

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import os
import glob
import zipfile
import pandas as pd

def find_file(candidates):
    for pat in candidates:
        matches = glob.glob(f'/content/drive/MyDrive/**/{pat}', recursive=True)
        if matches:
            return matches[0]
    return None

train_zip = find_file(['train_images.zip'])
train_csv_path = find_file(['train_metadata.csv'])
test_zip = find_file(['eval_images.zip', 'test_images.zip'])
test_csv_path = find_file(['test_metadata.csv', 'eval_metadata.csv'])

print(f"Train zip: {train_zip}")
print(f"Train csv: {train_csv_path}")
print(f"Test zip:  {test_zip}")
print(f"Test csv:  {test_csv_path}")

if not train_zip or not test_zip:
    raise FileNotFoundError("Check Google Drive: Ensure shortcuts for train_images.zip and eval_images.zip are added to My Drive.")

def extract_archive(zip_path, target_folder):
    os.makedirs(target_folder, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(target_folder)
    inner = [os.path.join(target_folder, d) for d in os.listdir(target_folder) if os.path.isdir(os.path.join(target_folder, d))]
    return inner[0] if len(inner) == 1 else target_folder

train_dir = extract_archive(train_zip, '/content/train_img')
test_dir = extract_archive(test_zip, '/content/test_img')

train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)

print(f"\nExtraction complete:")
print(f"Train: {len(train_df)} rows, images in {train_dir}")
print(f"Test:  {len(test_df)} rows, images in {test_dir}")

Train zip: /content/drive/MyDrive/train_images.zip
Train csv: /content/drive/MyDrive/train_metadata.csv
Test zip:  /content/drive/MyDrive/eval_images.zip
Test csv:  /content/drive/MyDrive/test_metadata.csv

Extraction complete:
Train: 7854 rows, images in /content/train_img/train_images
Test:  2000 rows, images in /content/test_img/eval_images


In [4]:
import cv2
import math
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class LunarStreamDataset(Dataset):
    def __init__(self, df, img_dir, is_train=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        name = str(row['image_id'])
        if not name.endswith(('.png', '.jpg', '.jpeg')):
            name += '.png'

        path = os.path.join(self.img_dir, name)
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(f"Missing {path}")

        # Grayscale normalized [-1, 1]
        img_t = torch.tensor((img.astype(np.float32) / 127.5) - 1.0).unsqueeze(0)

        # Sun direction vector: [sin(theta), cos(theta)]
        rad = math.radians(float(row['sun_azimuth_angle']))
        s_vec = torch.tensor([math.sin(rad), math.cos(rad)], dtype=torch.float32)

        if 'label' in row:
            return img_t, s_vec, torch.tensor(row['label'], dtype=torch.float32)
        return img_t, s_vec, str(row['image_id'])

class FiLMBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.fc = nn.Linear(2, channels * 2)

    def forward(self, x, c):
        g, b = torch.chunk(self.fc(c), 2, dim=1)
        return x * (1.0 + g.unsqueeze(-1).unsqueeze(-1)) + b.unsqueeze(-1).unsqueeze(-1)

class SunFiLMNet(nn.Module):
    def __init__(self):
        super().__init__()
        base = timm.create_model('resnet18', pretrained=True, in_chans=1)
        self.stem = nn.Sequential(base.conv1, base.bn1, base.act1, base.maxpool)
        self.l1, self.f1 = base.layer1, FiLMBlock(64)
        self.l2, self.f2 = base.layer2, FiLMBlock(128)
        self.l3, self.f3 = base.layer3, FiLMBlock(256)
        self.l4, self.f4 = base.layer4, FiLMBlock(512)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(514, 64),
            nn.SiLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x, c):
        x = self.stem(x)
        x = self.f1(self.l1(x), c)
        x = self.f2(self.l2(x), c)
        x = self.f3(self.l3(x), c)
        x = self.f4(self.l4(x), c)
        feat = self.pool(x).flatten(1)
        return self.head(torch.cat([feat, c], dim=1))

print(f"Model and dataset definitions loaded. Using device: {device}")

Model and dataset definitions loaded. Using device: cuda


In [5]:
from torch.utils.data import DataLoader
from sklearn.metrics import balanced_accuracy_score
from torch.optim import AdamW

# 85% train split, 15% validation split
val_split = train_df.sample(frac=0.15, random_state=42)
train_split = train_df.drop(val_split.index)

train_loader = DataLoader(LunarStreamDataset(train_split, train_dir, is_train=True), batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(LunarStreamDataset(val_split, train_dir, is_train=False), batch_size=64, shuffle=False)

model = SunFiLMNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=2e-4, weight_decay=1e-2)

best_bal_acc = 0.0

print("Training model across 5 epochs...")
for epoch in range(1, 6):
    model.train()
    running_loss = 0.0
    for imgs, s_vecs, labels in train_loader:
        imgs, s_vecs, labels = imgs.to(device), s_vecs.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        loss = criterion(model(imgs, s_vecs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Fast validation
    model.eval()
    probs, targets = [], []
    with torch.no_grad():
        for imgs, s_vecs, labels in val_loader:
            p = torch.sigmoid(model(imgs.to(device), s_vecs.to(device))).cpu().numpy().flatten()
            probs.extend(p)
            targets.extend(labels.numpy().flatten())

    probs, targets = np.array(probs), np.array(targets)
    for t in np.linspace(0.35, 0.65, 31):
        score = balanced_accuracy_score(targets, (probs >= t).astype(int))
        if score > best_bal_acc:
            best_bal_acc = score

    print(f"Epoch {epoch:02d}/05 | Loss: {running_loss/len(train_loader):.4f} | Val Bal Acc: {best_bal_acc*100:.2f}%")

torch.save(model.state_dict(), 'final_lunar_model.pth')
print("Training complete! Model saved.")

model.safetensors: reconstructing file:   0%|          |  0.00B / 46.8MB            

model.safetensors: downloading bytes:           |  0.00B            

Training model across 5 epochs...
Epoch 01/05 | Loss: 0.5312 | Val Bal Acc: 77.26%
Epoch 02/05 | Loss: 0.5082 | Val Bal Acc: 77.26%
Epoch 03/05 | Loss: 0.5045 | Val Bal Acc: 77.26%
Epoch 04/05 | Loss: 0.4961 | Val Bal Acc: 77.26%
Epoch 05/05 | Loss: 0.4898 | Val Bal Acc: 77.26%
Training complete! Model saved.


In [6]:
from torch.utils.data import DataLoader

test_loader = DataLoader(
    LunarStreamDataset(test_df, test_dir, is_train=False),
    batch_size=64,
    shuffle=False
)

model.eval()
test_ids = []
raw_test_probs = []

print("Running test set predictions...")
with torch.no_grad():
    for imgs, s_vecs, ids in test_loader:
        p = torch.sigmoid(model(imgs.to(device), s_vecs.to(device))).cpu().numpy().flatten()
        raw_test_probs.extend(p)
        test_ids.extend(ids)

raw_test_probs = np.array(raw_test_probs)
print(f"Predictions collected for {len(raw_test_probs)} evaluation images.")

Running test set predictions...
Predictions collected for 2000 evaluation images.


In [7]:
import numpy as np
import pandas as pd
from google.colab import files

# Balance threshold based on the dataset ratio (~36.4% negatives)
calibrated_threshold = np.percentile(raw_test_probs, 36.4)
print(f"Calculated optimal threshold: {calibrated_threshold:.4f}")

calibrated_preds = (raw_test_probs >= calibrated_threshold).astype(int)

# Create final submission dataframe
submission = pd.DataFrame({
    'image_id': test_ids,
    'label': calibrated_preds
})

sub_path = '/content/submission.csv'
submission.to_csv(sub_path, index=False)

print(f"\nSaved clean submission to: {sub_path}")
print("Predicted class distribution:")
print(submission['label'].value_counts())

# Download file directly to your browser
files.download(sub_path)

Calculated optimal threshold: 0.8329

Saved clean submission to: /content/submission.csv
Predicted class distribution:
label
1    1272
0     728
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>